In [1]:
import os, random
from pathlib import Path

mc_dir = Path("/volatile/clas12/zurek/SULI/mc_v01")
files = sorted(mc_dir.glob("*.root"))
sizes = {f.name: f.stat().st_size for f in files}

# Shuffle (fixed seed for reproducibility), then sort by size descending
random.seed(42)
shuffled = list(sizes.items())
random.shuffle(shuffled)
shuffled.sort(key=lambda x: -x[1])

# Greedy bin-packing into three buckets with target ratios 0.7 / 0.15 / 0.15
total = sum(sizes.values())
targets = {"train": 0.70 * total, "val": 0.15 * total, "test": 0.15 * total}
buckets = {"train": [], "val": [], "test": []}
loads   = {"train": 0,   "val": 0,   "test": 0}

for name, size in shuffled:
    # Assign to whichever split is most "under target" relative to its budget
    deficit = {k: targets[k] - loads[k] for k in buckets}
    pick = max(deficit, key=deficit.get)
    buckets[pick].append(name)
    loads[pick] += size

for split in ("train", "val", "test"):
    frac = loads[split] / total
    print(f"{split}: {len(buckets[split])} files, {loads[split]/1e9:.1f} GB, {frac:.3f} of total")
    Path("slurm").mkdir(exist_ok=True)
    with open(f"slurm/{split}_files.txt", "w") as f:
        f.write("\n".join(sorted(buckets[split])) + "\n")

train: 190 files, 66.5 GB, 0.701 of total
val: 55 files, 14.2 GB, 0.150 of total
test: 55 files, 14.2 GB, 0.150 of total
